## 1.计算流通市值

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

capital_dt = pd.read_parquet('data/capital.parquet')
capital_dt=capital_dt.sort_values(by=['stock_code', 'change_date'], ascending=[True, True])
capital_dt.rename(columns={'change_date': 'trade_date'}, inplace=True)
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
mv_factor_dt=pd.merge(price_dt, capital_dt, on=['stock_code', 'trade_date'], how='left')
mv_factor_dt.fillna(method='ffill', inplace=True)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares
0,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466,NaN,NaN,NaN,NaN
1,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876,NaN,NaN,NaN,NaN
2,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937,NaN,NaN,NaN,NaN
3,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831,NaN,NaN,NaN,NaN
4,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326,NaN,NaN,NaN,NaN


In [9]:
mv_factor_dt['float_a_shares_mv']=mv_factor_dt['float_a_shares']*mv_factor_dt['close_price']
mv_factor_dt.tail()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551


## 2.插入行业分类信息,计算市值因子
*流通市值 = close_price * float_a_shares*

(小市值策略应该看A股流通市值，直接跟盘子相关)

*市值因子 = -Ln(流通市值) 对于行业做标准化*

In [ ]:
cat_df=pd.read_parquet('data/ref_data/Stock_Industry_Year.parquet')
mv_factor_dt['industry_name'] = mv_factor_dt.assign(year=pd.to_datetime(mv_factor_dt['trade_date'].astype(str)).dt.year).merge(cat_df[['stock_code', 'year', 'industry_name']], on=['stock_code', 'year'], how='left')['industry_name']

In [11]:
import numpy as np
mv_factor_dt.dropna(inplace=True)
mv_factor_dt['mv']=-np.log(mv_factor_dt['float_a_shares_mv'])
mv_factor_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,industry_name,mv
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,银行,-15.593639
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,银行,-15.628187
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,银行,-15.530255


In [12]:
# 按「交易日期」分组，对当天全行业的mv列做Z-score标准化（(值-均值)/标准差）
mv_factor_dt['mv_standardized'] = mv_factor_dt.groupby('trade_date')['mv'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,industry_name,mv,mv_standardized
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,银行,-15.593639,-1.744443
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038,-1.750973
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,银行,-15.628187,-1.774256
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038,-1.757097
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,银行,-15.530255,-1.726456


In [13]:
mv_factor_dt.tail()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,industry_name,mv,mv_standardized
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613,医药生物,-11.350206,1.797748
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320,医药生物,-11.356876,1.805374
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847,医药生物,-11.342929,1.809709
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964,医药生物,-11.334466,1.821782
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551,医药生物,-11.330492,1.837061


## 3.股票池筛选与缩尾
遍历每个日期，从中证1000基金持仓中遴选小盘股，可以基本剔除了严重问题股、流动性枯竭股，收益温和，回撤更小。

所有样本在当日排序后winsorize，取1%分位数作为下限，99%分位数作为上限。然后为了分散风险，每个行业选stock_num只股票生成一个选股的df


In [14]:
from scipy.stats.mstats import winsorize  # 专业的缩尾处理函数
from datetime import datetime, timedelta

stock_num = 5
# 读取数据（确保文件路径正确）
fund_df = pd.read_parquet('data/ref_data/ETF_hold_512100.SH.parquet')

# ===================== 1. 数据预处理 =====================
# 处理mv_factor_dt的日期格式（数字转日期）
# 注：原代码中mv_factor_dt未定义，需确保该DataFrame已提前加载
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))

# 处理基金持仓数据的日期格式
fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})  # 重命名避免混淆
fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

# 生成基金持仓的有效时间区间（半年报：1-6月，年报：7-12月）
def get_position_valid_period(row):
    """计算基金持仓的有效时间区间"""
    year = row['report_year']
    if row['report_type'] == '中报':
        start_date = pd.to_datetime(f'{year}-01-01')
        end_date = pd.to_datetime(f'{year}-06-30')
    elif row['report_type'] == '年报':
        start_date = pd.to_datetime(f'{year}-07-01')
        end_date = pd.to_datetime(f'{year}-12-31')
    else:
        start_date = end_date = row['report_end_date']
    return pd.Series([start_date, end_date], index=['pos_start_date', 'pos_end_date'])

# 给基金持仓添加有效时间区间
fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)

# 提取中证1000基金持仓的股票代码+有效区间（去重）
fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

# ===================== 2. 核心选股逻辑 =====================
def select_stocks_by_date(date_group):
    """
    对单个日期的样本执行选股：
    1. 筛选当日属于中证1000持仓的股票
    2. 对mv列做1%/99%缩尾
    3. 按行业分组，每个独特行业选5只（该行业内缩尾后MV最高的）
    """
    # 获取当日日期和数据
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 步骤1：筛选当日属于中证1000基金持仓的股票
    valid_holdings = fund_holdings[
        (fund_holdings['pos_start_date'] <= trade_date) & 
        (fund_holdings['pos_end_date'] >= trade_date)
    ]['stock_code'].tolist()
    holding_stocks = daily_data[daily_data['stock_code'].isin(valid_holdings)]
    
    if len(holding_stocks) == 0:
        return pd.DataFrame()  # 无持仓股票则返回空
    
    # 步骤2：对mv列做1%（下限）、99%（上限）的缩尾处理
    holding_stocks['mv_winsorized'] = winsorize(
        holding_stocks['mv_standardized'].values, 
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 步骤3：按行业分组，每个行业选缩尾后MV最高的1只（核心修改点）
    # 过滤掉行业名称为空/NaN的记录（避免无意义分组）
    holding_stocks = holding_stocks.dropna(subset=['industry_name'])
    # 按行业分组，每组取mv_winsorized最大的n只
    selected = holding_stocks.groupby('industry_name', group_keys=False).apply(
        lambda x: x.sort_values('mv_winsorized', ascending=False).head(stock_num)
    )
    
    # 添加选股标识：按缩尾后MV降序排名（可选：也可按行业名排序）
    selected = selected.sort_values('mv_winsorized', ascending=False)
    selected['selection_rank'] = range(1, len(selected)+1)  # 排名按整体MV从高到低
    
    return selected

# 按交易日分组执行选股
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引（清理groupby后的多级索引）
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 'industry_name', 'mv', 'mv_winsorized', 'selection_rank']]
final_selection_df.tail()

,trade_date,stock_code,industry_name,mv,mv_winsorized,selection_rank
327695,2025-06-30,000712.SZ,非银金融,-13.977728,-0.621403,148.0
327696,2025-06-30,600643.SH,非银金融,-13.980844,-0.624148,149.0
327697,2025-06-30,001203.SZ,钢铁,-14.015449,-0.654625,150.0
327698,2025-06-30,600864.SH,非银金融,-14.032331,-0.669493,151.0
327699,2025-06-30,002408.SZ,石油石化,-14.057202,-0.691397,152.0


In [16]:
final_selection_df.to_excel('records/mv_selection.xlsx', index=False)

In [17]:
import pandas as pd
final_selection_df=pd.read_excel('records/mv_selection.xlsx')
final_selection_df.head()

,trade_date,stock_code,industry_name,mv,mv_winsorized,selection_rank
0,2016-07-01,002726.SZ,食品饮料,-11.987691,1.396203,1
1,2016-07-01,002713.SZ,建筑装饰,-12.107389,1.396203,2
2,2016-07-01,002346.SZ,建筑材料,-11.933907,1.396203,3
3,2016-07-01,603166.SH,汽车,-11.814988,1.396203,4
4,2016-07-01,603398.SH,轻工制造,-12.180835,1.396203,5


In [18]:
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
price_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover
2983,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466
8347,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876
10530,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937
13553,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831
10100,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326


## 小市值单因子回测
基金持仓的年报数据最早从2016年7月1日开始。鉴于此，回测区间为2016年7月1日到2025年6月30日
交易日收盘时选股，需要次交易日以复权收盘价执行买入。每20个交易日以复权收盘价卖出调整一次持仓，收盘时按照复权收盘价买入，进行下一个循环

In [43]:
import pandas as pd
import numpy as np
from util import (load_and_preprocess_price, load_selection,
                  get_top_n_selection, compute_benchmark_returns,
                  compute_metrics, plot_results)

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000        # initial capital
START_DATE = '2020-01-01'       # backtest start date (inclusive)
END_DATE   = '2025-06-30'       # backtest end date (inclusive)
REBALANCE_FREQ = 5             # rebalance every N calendar days (using available selection dates)
TOP_N = 10                       # number of top‑ranked stocks to hold each rebalance

PRICE_FILE = 'data/eod_prices.parquet'   # path to price file
SELECTION_FILE = 'records/mv_selection.xlsx'  # path to selection file

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)
selection_df = load_selection(SELECTION_FILE)

# Create pivot table: rows = dates, columns = stocks, values = adjusted_close
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
# Ensure index is datetime (fix for categorical index)
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Get sorted unique selection dates (these are the only dates we consider for rebalancing)
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available in the given date range.")

# Determine rebalance dates (every REBALANCE_FREQ-th date in the selection_dates list)
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# Generate the full timeline (all trading days in the pivot index)
all_dates = price_pivot.index.tolist()
# Ensure we start from the first rebalance date (before that we have no positions)
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# ----------------------------
# Backtest loop
# ----------------------------
cash = INITIAL_CASH
holdings = {}  # stock_code -> shares
portfolio_values = []
benchmark_cumulative = None

# Pre‑compute benchmark returns (daily equal‑weighted average)
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# For each day in timeline
for date in timeline:
    # Get today's prices for all stocks (if missing, price will be NaN)
    today_prices = price_pivot.loc[date]

    # --- Rebalance if today is a rebalance date ---
    if date in rebalance_dates:
        # 1. Sell all current holdings
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash += shares * price
            # If price is missing, we assume the stock is worthless (sell for 0)
        holdings = {}

        # 2. Get selected stocks for today
        selected = get_top_n_selection(selection_df, date, TOP_N)

        # 3. Buy new holdings equally
        if selected:
            # Filter stocks that have a valid price today
            valid_stocks = [s for s in selected if pd.notna(today_prices.get(s, np.nan))]
            if valid_stocks:
                amount_per_stock = cash / len(valid_stocks)
                for stock in valid_stocks:
                    price = today_prices[stock]
                    shares = amount_per_stock / price
                    holdings[stock] = shares
                cash = 0.0  # all cash invested
            else:
                # No valid stocks – keep cash
                pass

    # --- Calculate total portfolio value today ---
    portfolio_value = cash
    for stock, shares in holdings.items():
        price = today_prices.get(stock, np.nan)
        if pd.notna(price):
            portfolio_value += shares * price
        # If price is missing, the position contributes 0
    portfolio_values.append(portfolio_value)

# ----------------------------
# Prepare results
# ----------------------------
portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')

benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
benchmark_funds = INITIAL_CASH * benchmark_series

portfolio_returns = portfolio_series.pct_change().dropna()

metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)
print("\n--- Performance Metrics ---")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

plot_results(portfolio_series, benchmark_funds, timeline,
             title='Backtest: Top {} Stocks Rebalanced Every {} Days'.format(TOP_N, REBALANCE_FREQ))

Loading data...
Number of rebalance dates: 266

--- Performance Metrics ---
Annualized Return: 0.0799
Volatility: 0.2958
Sharpe Ratio: 0.2701
Max Drawdown: -0.3828


## 纯小市值因子回测官方参考
![纯小市值因子回测参考](docs/纯小市值-参考.png)

## 优化方向
观察发现调仓频率\选股数量对收益影响很大；
区分测试集与训练集跑回测；
纯小市值因子不够有效，加入动量和成长因子搭配是否更好……
现在只有收盘价，调仓必须等待一个交易日，有信息衰减。

In [40]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import warnings
warnings.filterwarnings('ignore')

# ==================== 参数配置 ====================
STOCK_NUM_PER_INDUSTRY = 5          # 每个行业选股数量
WINDSORIZE_LIMITS = (0.01, 0.01)    # 缩尾比例
START_DATE = '2016-01-01'
END_DATE = '2025-06-30'
PRICE_FILE = 'data/eod_prices.parquet'               # 价格数据路径
FUND_HOLD_FILE = 'data/ref_data/ETF_hold_512100.SH.parquet'  # 基金持仓路径
OUTPUT_FILE = 'records/mv_selection_v2.xlsx'         # 输出文件

# ==================== 数据加载 ====================
print("加载数据...")

# 1. 价格数据（用于计算动量）
price_df = pd.read_parquet(PRICE_FILE)
price_df['trade_date'] = pd.to_datetime(price_df['trade_date'])
price_df = price_df.sort_values(['stock_code', 'trade_date'])

# 2. 基金持仓数据
fund_df = pd.read_parquet(FUND_HOLD_FILE)
fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})
fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'])
mv_factor_dt = mv_factor_dt[(mv_factor_dt['trade_date'] >= START_DATE) & (mv_factor_dt['trade_date'] <= END_DATE)]

# ==================== 基金持仓时间区间处理 ====================
def get_position_valid_period(row):
    """计算持仓报告的有效区间（中报：1-6月，年报：7-12月）"""
    year = row['report_year']
    if row['report_type'] == '中报':
        start = pd.to_datetime(f'{year}-01-01')
        end = pd.to_datetime(f'{year}-06-30')
    elif row['report_type'] == '年报':
        start = pd.to_datetime(f'{year}-07-01')
        end = pd.to_datetime(f'{year}-12-31')
    else:
        start = end = row['report_end_date']
    return pd.Series([start, end], index=['pos_start_date', 'pos_end_date'])

fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)
fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

# ==================== 动量因子计算 ====================
print("计算动量因子...")
# 计算30日收益率（pct_change的periods=30）
price_df['return_30'] = price_df.groupby('stock_code')['adjusted_close'].pct_change(30)
# 计算90日收益率（可选，用于中期过滤）
price_df['return_90'] = price_df.groupby('stock_code')['adjusted_close'].pct_change(90)

# 每日对收益率进行百分位排名（0~1），涨幅越大排名越高
price_df['rank_mom_30'] = price_df.groupby('trade_date')['return_30'].rank(pct=True, ascending=True)
price_df['rank_mom_90'] = price_df.groupby('trade_date')['return_90'].rank(pct=True, ascending=True)

# 合并动量因子到 mv_factor_dt
# 确保类型一致
price_df['stock_code'] = price_df['stock_code'].astype(str)
mv_factor_dt['stock_code'] = mv_factor_dt['stock_code'].astype(str)

# 关键修复：强制两个trade_date列均为datetime64[ns]类型
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'], errors='coerce')
price_df['trade_date'] = pd.to_datetime(price_df['trade_date'], errors='coerce')
# 删除转换失败的行（如果有）
mv_factor_dt = mv_factor_dt.dropna(subset=['trade_date'])
price_df = price_df.dropna(subset=['trade_date'])

# 现在合并
mv_factor_dt = mv_factor_dt.merge(price_df[['stock_code', 'trade_date', 'return_30', 'rank_mom_30', 'rank_mom_90']],
                                  on=['stock_code', 'trade_date'], how='left')

# 处理动量因子缺失值（如上市不足30天的股票），用0填充，后续在选股时会自然排后面
mv_factor_dt['rank_mom_30'] = mv_factor_dt['rank_mom_30'].fillna(0)
mv_factor_dt['rank_mom_90'] = mv_factor_dt['rank_mom_90'].fillna(0)

# ==================== 选股函数 ====================
def select_stocks_by_date(date_group):
    """对单个交易日执行选股"""
    trade_date = date_group.name
    daily_data = date_group.copy()

    # 1. 筛选当日基金持仓的股票
    valid_stocks = fund_holdings[
        (fund_holdings['pos_start_date'] <= trade_date) & 
        (fund_holdings['pos_end_date'] >= trade_date)
    ]['stock_code'].tolist()
    candidate = daily_data[daily_data['stock_code'].isin(valid_stocks)]
    if candidate.empty:
        return pd.DataFrame()

    # 2. 对市值因子缩尾（1%/99%）
    candidate['mv_winsorized'] = winsorize(candidate['mv_standardized'].values,
                                           limits=WINDSORIZE_LIMITS,
                                           inclusive=(True, True))

    # 3. 去掉行业为空的数据
    candidate = candidate.dropna(subset=['industry_name'])

    # 4. 计算排名（小市值：mv_winsorized 越小越好，所以 rank_mv 升序）
    candidate['rank_mv'] = candidate.groupby('industry_name')['mv_winsorized'].rank(ascending=True)

    # 5. 短期动量排名（rank_mom_30 越大越好，所以 rank_mom 降序）
    candidate['rank_mom'] = candidate.groupby('industry_name')['rank_mom_30'].rank(ascending=False)

    # 6. 中期动量排名（可选，此处也降序，作为复合因子的一部分）
    candidate['rank_mid'] = candidate.groupby('industry_name')['rank_mom_90'].rank(ascending=False)

    # 7. 复合因子（小市值 + 短期动量 + 中期动量）
    # 权重可调，这里等权相加
    candidate['composite'] = candidate['rank_mv'] + candidate['rank_mom'] + candidate['rank_mid']

    # 8. 每个行业选取 composite 最小的 STOCK_NUM_PER_INDUSTRY 只股票
    selected = candidate.groupby('industry_name', group_keys=False).apply(
        lambda x: x.nsmallest(STOCK_NUM_PER_INDUSTRY, 'composite')
    )

    # 9. 整体排名（按 composite 升序）
    selected = selected.sort_values('composite')
    selected['selection_rank'] = range(1, len(selected) + 1)

    return selected

# ==================== 执行选股 ====================
print("开始逐日选股...")
selected_stocks = mv_factor_dt.groupby('trade_date', group_keys=False).apply(select_stocks_by_date)

# 重置索引，去除多余索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保存结果
selected_stocks.to_excel(OUTPUT_FILE, index=False)
print(f"选股完成！结果保存至 {OUTPUT_FILE}")
print(f"共选出 {len(selected_stocks)} 条记录，涉及 {selected_stocks['trade_date'].nunique()} 个交易日。")

加载数据...
计算动量因子...
开始逐日选股...
选股完成！结果保存至 records/mv_selection_v2.xlsx
共选出 327700 条记录，涉及 2184 个交易日。


In [42]:
import pandas as pd
import numpy as np
from util import (load_and_preprocess_price, load_selection,
                  get_top_n_selection, compute_benchmark_returns,
                  compute_metrics, plot_results)

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000        # initial capital
START_DATE = '2020-01-01'       # backtest start date (inclusive)
END_DATE   = '2025-06-30'       # backtest end date (inclusive)
REBALANCE_FREQ = 5             # rebalance every N calendar days (using available selection dates)
TOP_N = 10                       # number of top‑ranked stocks to hold each rebalance

PRICE_FILE = 'data/eod_prices.parquet'   # path to price file
SELECTION_FILE = 'records/mv_selection_v2.xlsx'  # path to selection file

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)
selection_df = load_selection(SELECTION_FILE)

# Create pivot table: rows = dates, columns = stocks, values = adjusted_close
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
# Ensure index is datetime (fix for categorical index)
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Get sorted unique selection dates (these are the only dates we consider for rebalancing)
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available in the given date range.")

# Determine rebalance dates (every REBALANCE_FREQ-th date in the selection_dates list)
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# Generate the full timeline (all trading days in the pivot index)
all_dates = price_pivot.index.tolist()
# Ensure we start from the first rebalance date (before that we have no positions)
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# ----------------------------
# Backtest loop
# ----------------------------
cash = INITIAL_CASH
holdings = {}  # stock_code -> shares
portfolio_values = []
benchmark_cumulative = None

# Pre‑compute benchmark returns (daily equal‑weighted average)
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# For each day in timeline
for date in timeline:
    # Get today's prices for all stocks (if missing, price will be NaN)
    today_prices = price_pivot.loc[date]

    # --- Rebalance if today is a rebalance date ---
    if date in rebalance_dates:
        # 1. Sell all current holdings
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash += shares * price
            # If price is missing, we assume the stock is worthless (sell for 0)
        holdings = {}

        # 2. Get selected stocks for today
        selected = get_top_n_selection(selection_df, date, TOP_N)

        # 3. Buy new holdings equally
        if selected:
            # Filter stocks that have a valid price today
            valid_stocks = [s for s in selected if pd.notna(today_prices.get(s, np.nan))]
            if valid_stocks:
                amount_per_stock = cash / len(valid_stocks)
                for stock in valid_stocks:
                    price = today_prices[stock]
                    shares = amount_per_stock / price
                    holdings[stock] = shares
                cash = 0.0  # all cash invested
            else:
                # No valid stocks – keep cash
                pass

    # --- Calculate total portfolio value today ---
    portfolio_value = cash
    for stock, shares in holdings.items():
        price = today_prices.get(stock, np.nan)
        if pd.notna(price):
            portfolio_value += shares * price
        # If price is missing, the position contributes 0
    portfolio_values.append(portfolio_value)

# ----------------------------
# Prepare results
# ----------------------------
portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')

benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
benchmark_funds = INITIAL_CASH * benchmark_series

portfolio_returns = portfolio_series.pct_change().dropna()

metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)
print("\n--- Performance Metrics ---")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

plot_results(portfolio_series, benchmark_funds, timeline,
             title='Backtest: Top {} Stocks Rebalanced Every {} Days'.format(TOP_N, REBALANCE_FREQ))

Loading data...
Number of rebalance dates: 266

--- Performance Metrics ---
Annualized Return: -0.1613
Volatility: 0.3102
Sharpe Ratio: -0.5200
Max Drawdown: -0.8117


代码是AI写的，不知道为什么动量因子到2022年以后失效了